# Supplementary Figure 2 | Concordance of stranded and unstranded dGTEx quantification by annotation version

**a.** Joint distribution of the fraction of exonic sequence that overlaps another gene (% ambiguous) in GENCODE v39 versus v47 for protein-coding, lncRNA, and pseudogene genes (each gene is a point; marginal histograms show distribution in each version individually). Grey shading marks genes that newly become highly ambiguous in v47 (>75% ambiguous in v47, ≤75% in v39).

**b.** Distribution of the change in % ambiguous (v47−v39) by biotype. Overall, most genes show little change. Although ambiguity can increase or decrease with the updated annotation, increased ambiguity is more common, reflecting addition of thousands of new genes, some of which overlap previously annotated genes.

**c.** Hexbin density of sample×gene pairs comparing v39 stranded versus v39 unstranded dGTEx RNA-SeQC log10(TPM+1) for protein-coding, lncRNA, and pseudogene genes. Overall Lin’s concordance correlation coefficient (CCC) across pooled sample×gene pairs is indicated (CCC = 0.995).

**d.** Same as (c) for v47 stranded versus v47 unstranded quantification dGTEx RNA-SeQC log10(TPM+1). Genome-wide concordance remains high under the updated annotation (CCC = 0.980).

**e.** Per-gene Lin’s CCC (across samples) between matched stranded and unstranded TPM versus % ambiguous exonic sequence, colored by GENCODE version and stratified by stranded median TPM < 1 (left) vs TPM > 1 (right). Each point is an individual gene; line and ribbon are rolling median and IQR. Vertical dashed line marks 75% ambiguous. Lowly expressed genes have noisy quantifications and are more discordant across ambiguity levels. For genes with median TPM > 1, concordance remains high until ~75% ambiguity, then falls sharply.

**f.** Each point is a gene with unchanged annotation coordinates between versions that newly become highly ambiguous in v47 (n=387 genes with >75% ambiguous in v47, ≤75% in v39), per-gene CCC across samples is plotted for stranded vs unstranded using GENCODE v39 versus v47. Points are colored by biotype, diagonal is y=x. Most newly highly ambiguous genes have lower concordance quantified with v47 than with v39.


## Setup


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.patches import Rectangle

plt.rc("axes", labelsize=6)
plt.rc("xtick", labelsize=6)
plt.rc("ytick", labelsize=6)
plt.rc("legend", fontsize=5)
plt.rc("legend", title_fontsize=6)
plt.rc("font", size=6)
plt.rc("figure", titlesize=7)
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["figure.figsize"] = (7.2, 7.4)
plt.rcParams["figure.dpi"] = 300
plt.rcParams["savefig.dpi"] = 300

fig_width, fig_height, fig_dpi = 7.2, 7.4, 300
high_amb_pct = 75.0
median_tpm_cut = 1.0
rolling_window = 201
rolling_min_periods = 40
ccc_gene_types = ["protein_coding", "lncRNA", "pseudogene"]
expr_bin_labels = ["< 1", "> 1"]
ccc_color_v47 = "#4b0082"  # indigo
ccc_color_v39 = "#e69f00"  # amber

gene_type_pretty = {
    "lncRNA": "lncRNA",
    "protein_coding": "protein-coding",
    "pseudogene": "pseudogene",
}
gene_type_palette = {
    "lncRNA": sns.color_palette("Paired", 12)[5],
    "protein_coding": sns.color_palette("Paired", 12)[1],
    "pseudogene": sns.color_palette("Paired", 12)[3],
}


## Paths

In [ ]:
gencode_class_path = Path("path/to/gencode_v39_v47_gene_classification.txt")
dgtex_meta_path = Path("path/to/dGTEx_Analysis_2025-07-21_v1_RNA_Metadata_subset_EXCLUSIONS.xlsx")
# Official dGTEx RNA-SeQC = stranded GENCODE v47
stranded_v47_tpm_path = Path("path/to/dGTEx_Analysis_2025-07-21_v1_RNASeQCv2.4.3_gene_tpm.gct.gz")
stranded_v39_tpm_path = Path("path/to/dGTEx_Analysis_2026-10-31_v1_stranded_v39_collapsed_RNASeQCv2.4.3_gene_tpm.parquet")
unstranded_v47_tpm_path = Path("path/to/dGTEx_Analysis_2025-11-13_v1_unstranded_collapsed_RNASeQCv2.4.3_gene_tpm.parquet")
unstranded_v39_tpm_path = Path("path/to/dGTEx_Analysis_2026-03-10_v1_unstranded_v39_collapsed_RNASeQCv2.4.3_gene_tpm.parquet")

fig_dir = Path("figures")
fig_pdf = fig_dir / "supp_figure_2_ccc_by_ambiguity.pdf"
fig_png = fig_dir / "supp_figure_2_ccc_by_ambiguity.png"


## Load data


### GENCODE classification


In [ ]:
gencode_genes = pd.read_table(gencode_class_path, low_memory=False)
# Classification file splits lncRNA into novel/existing; this figure pools them.
gencode_genes["gene_type_grouped"] = gencode_genes["gene_type_grouped"].replace(
    {"novel_lncRNA": "lncRNA", "existing_lncRNA": "lncRNA"}
)
print(f"GENCODE genes: {len(gencode_genes):,}")
print(gencode_genes["gene_type_grouped"].value_counts().to_string())


### dGTEx sample metadata


In [ ]:
dgtex_sample_meta = pd.read_excel(dgtex_meta_path)
passed = list(
    dgtex_sample_meta.loc[
        dgtex_sample_meta["EXCLUDE from Analysis Freeze"] != "EXCLUDE",
        "Sample ID (Data Release Official ID)",
    ].values
)
print(f"passed dGTEx samples: {len(passed)}")


### dGTEx stranded v47 TPM


In [ ]:
print(f"Loading stranded v47 (official dGTEx): {stranded_v47_tpm_path}")
df = pd.read_table(stranded_v47_tpm_path, skiprows=2)
df = df[["Name", *passed]].set_index("Name").T
df.columns = df.columns.str.replace(r"\.\d+$", "", regex=True)
tpm_stranded_v47 = df.T.groupby(level=0).sum().T
print(f"  stranded v47 shape: {tpm_stranded_v47.shape}")


### dGTEx stranded v39 TPM


In [ ]:
print(f"Loading stranded v39: {stranded_v39_tpm_path}")
df = pd.read_parquet(stranded_v39_tpm_path)
df = df[["Name", *passed]].set_index("Name").T
df.columns = df.columns.str.replace(r"\.\d+$", "", regex=True)
tpm_stranded_v39 = df.T.groupby(level=0).sum().T
print(f"  stranded v39 shape: {tpm_stranded_v39.shape}")


### dGTEx v47 unstranded TPM


In [ ]:
print(f"Loading v47 unstranded: {unstranded_v47_tpm_path}")
df = pd.read_parquet(unstranded_v47_tpm_path)
df = df[["Name", *passed]].set_index("Name").T
df.columns = df.columns.str.replace(r"\.\d+$", "", regex=True)
tpm_v47 = df.T.groupby(level=0).sum().T
print(f"  v47 shape: {tpm_v47.shape}")


### dGTEx v39 unstranded TPM


In [ ]:
print(f"Loading v39 unstranded: {unstranded_v39_tpm_path}")
df = pd.read_parquet(unstranded_v39_tpm_path)
df = df[["Name", *passed]].set_index("Name").T
df.columns = df.columns.str.replace(r"\.\d+$", "", regex=True)
tpm_v39 = df.T.groupby(level=0).sum().T
print(f"  v39 shape: {tpm_v39.shape}")


## Run analysis


### Align samples/genes and log TPM matrices


In [ ]:
samples_common = sorted(
    set(tpm_stranded_v47.index)
    & set(tpm_stranded_v39.index)
    & set(tpm_v47.index)
    & set(tpm_v39.index)
)
genes_common = sorted(
    set(tpm_stranded_v47.columns)
    & set(tpm_stranded_v39.columns)
    & set(tpm_v47.columns)
    & set(tpm_v39.columns)
)
print(f"common samples: {len(samples_common):,}; common genes: {len(genes_common):,}")

tpm_stranded_v47_unified = tpm_stranded_v47.loc[samples_common, genes_common]
tpm_stranded_v39_unified = tpm_stranded_v39.loc[samples_common, genes_common]
tpm_v47_unified = tpm_v47.loc[samples_common, genes_common]
tpm_v39_unified = tpm_v39.loc[samples_common, genes_common]

stranded_v47_mat = np.log10(tpm_stranded_v47_unified.values + 1)
stranded_v39_mat = np.log10(tpm_stranded_v39_unified.values + 1)
v47_mat = np.log10(tpm_v47_unified.values + 1)
v39_mat = np.log10(tpm_v39_unified.values + 1)
all_genes_arr = np.array(genes_common)
unstranded_mat = {"v39": v39_mat, "v47": v47_mat}


def _matched_stranded(version):
    if version == "v47":
        return stranded_v47_mat
    if version == "v39":
        return stranded_v39_mat
    raise KeyError(version)


### Per-gene CCC and expression strata


In [ ]:
def _batch_ccc(X, Y):
    """Per-gene Lin's CCC across samples (X, Y: samples x genes) -> one value per gene."""
    mu_x = X.mean(axis=0)
    mu_y = Y.mean(axis=0)
    Xc = X - mu_x
    Yc = Y - mu_y
    var_x = (Xc ** 2).mean(axis=0)
    var_y = (Yc ** 2).mean(axis=0)
    cov = (Xc * Yc).mean(axis=0)
    denom = var_x + var_y + (mu_x - mu_y) ** 2
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(denom > 0, 2 * cov / denom, np.nan)


gene_ccc_all = {
    "v47": _batch_ccc(v47_mat, stranded_v47_mat),
    "v39": _batch_ccc(v39_mat, stranded_v39_mat),
}

gene_median_tpm = pd.Series(
    np.median(tpm_stranded_v47_unified.values, axis=0),
    index=genes_common,
    name="median_tpm",
)
gene_expr = gene_median_tpm.to_frame()
_meta = gencode_genes.loc[
    gencode_genes["gene_type_grouped"].isin(ccc_gene_types),
    ["gene_id_stable", "gene_type_grouped", "pct_ambiguous_v47", "pct_ambiguous_v39", "change_level"],
].drop_duplicates("gene_id_stable")
gene_expr = gene_expr.join(_meta.set_index("gene_id_stable"), how="inner")
_tpm = gene_expr["median_tpm"]
gene_expr["expr_bin"] = pd.Categorical(
    np.where(_tpm < median_tpm_cut, expr_bin_labels[0], expr_bin_labels[1]),
    categories=expr_bin_labels,
    ordered=True,
)
print("expression bin counts (stranded median TPM):")
print(gene_expr["expr_bin"].value_counts().sort_index().to_string())

gene_ccc = gene_expr.reset_index(names="gene_id_stable").assign(
    ccc_v47=lambda d: gene_ccc_all["v47"][np.searchsorted(all_genes_arr, d["gene_id_stable"].to_numpy())],
    ccc_v39=lambda d: gene_ccc_all["v39"][np.searchsorted(all_genes_arr, d["gene_id_stable"].to_numpy())],
    pct_ambig_v47=lambda d: d["pct_ambiguous_v47"].astype(float) * 100.0,
    pct_ambig_v39=lambda d: d["pct_ambiguous_v39"].astype(float) * 100.0,
)
gene_ccc["newly_high_amb"] = (
    (gene_ccc["pct_ambig_v47"] > high_amb_pct)
    & (gene_ccc["pct_ambig_v39"] <= high_amb_pct)
    & (gene_ccc["change_level"] == "no_change")
)
print(
    f"per-gene CCC table: {len(gene_ccc):,} genes; "
    f"median CCC v47={np.nanmedian(gene_ccc['ccc_v47']):.3f}"
)
_nh = gene_ccc.loc[gene_ccc["newly_high_amb"]]
print(
    f"panel f newly highly ambiguous + no_change "
    f"(v47>{high_amb_pct:g}%, v39≤{high_amb_pct:g}%): "
    f"n={len(_nh):,}; median CCC v39={np.nanmedian(_nh['ccc_v39']):.3f}, "
    f"v47={np.nanmedian(_nh['ccc_v47']):.3f}"
)


### Sample×gene indices for hexbin scatters


In [ ]:
# Hexbin / pooled CCC: protein-coding, lncRNA, pseudogene genes with finite v47 ambiguity
_biotype_genes = set(
    gencode_genes.loc[
        gencode_genes["gene_type_grouped"].isin(ccc_gene_types)
        & gencode_genes["pct_ambiguous_v47"].notna(),
        "gene_id_stable",
    ]
)
_hex_genes = [g for g in genes_common if g in _biotype_genes]
scatter_gene_idx_pooled = (
    np.searchsorted(all_genes_arr, np.array(_hex_genes))
    if _hex_genes else np.array([], dtype=int)
)


def _ccc_sample_gene_pairs(X, Y):
    x = np.asarray(X, dtype=float).ravel()
    y = np.asarray(Y, dtype=float).ravel()
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]
    if len(x) < 2:
        return np.nan
    mu_x, mu_y = x.mean(), y.mean()
    var_x = ((x - mu_x) ** 2).mean()
    var_y = ((y - mu_y) ** 2).mean()
    cov = ((x - mu_x) * (y - mu_y)).mean()
    denom = var_x + var_y + (mu_x - mu_y) ** 2
    return float(2 * cov / denom) if denom > 0 else np.nan


ccc_pooled = {"v39": np.nan, "v47": np.nan}
if len(scatter_gene_idx_pooled):
    idx = scatter_gene_idx_pooled
    ccc_pooled["v47"] = _ccc_sample_gene_pairs(v47_mat[:, idx], stranded_v47_mat[:, idx])
    ccc_pooled["v39"] = _ccc_sample_gene_pairs(v39_mat[:, idx], stranded_v39_mat[:, idx])
print(
    f"pooled CCC genes: {len(scatter_gene_idx_pooled):,}; "
    f"v39={ccc_pooled['v39']:.3f}; v47={ccc_pooled['v47']:.3f}"
)


### Ambiguity table for panels a and b


In [ ]:
# Ambiguity table for panels a and b
amb_scatter = gencode_genes.loc[
    gencode_genes["gene_type_grouped"].isin(ccc_gene_types),
    ["gene_id_stable", "gene_type_grouped", "pct_ambiguous_v39", "pct_ambiguous_v47"],
].copy()
amb_scatter["pct_ambiguous_v39"] = amb_scatter["pct_ambiguous_v39"].astype(float)
amb_scatter["pct_ambiguous_v47"] = amb_scatter["pct_ambiguous_v47"].astype(float)
amb_scatter = amb_scatter.dropna(subset=["pct_ambiguous_v39", "pct_ambiguous_v47"])
amb_scatter["delta_ambig"] = amb_scatter["pct_ambiguous_v47"] - amb_scatter["pct_ambiguous_v39"]
amb_scatter["x"] = amb_scatter["pct_ambiguous_v39"] * 100
amb_scatter["y"] = amb_scatter["pct_ambiguous_v47"] * 100

delta_bin_edges = np.arange(-1.0, 1.0 + 1e-9, 0.05)
delta_bin_centers = 0.5 * (delta_bin_edges[:-1] + delta_bin_edges[1:]) * 100.0
delta_bin_labels = [
    f"({lo * 100:g}, {hi * 100:g}]" for lo, hi in zip(delta_bin_edges[:-1], delta_bin_edges[1:])
]
amb_scatter["delta_bin"] = pd.cut(
    amb_scatter["delta_ambig"],
    bins=delta_bin_edges,
    labels=delta_bin_labels,
    include_lowest=True,
)
print(f"amb_scatter genes: {len(amb_scatter):,}")


## Make figure


In [ ]:
def _rolling_ccc_trend(x, y, window=rolling_window, min_periods=rolling_min_periods):
    """Rolling median + IQR of y along x-sorted genes."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]
    if len(x) < min_periods:
        return np.array([]), np.array([]), np.array([]), np.array([])
    order = np.argsort(x)
    xs = x[order]
    s = pd.Series(y[order])
    med = s.rolling(window=window, center=True, min_periods=min_periods).median().to_numpy()
    q25 = s.rolling(window=window, center=True, min_periods=min_periods).quantile(0.25).to_numpy()
    q75 = s.rolling(window=window, center=True, min_periods=min_periods).quantile(0.75).to_numpy()
    return xs, med, q25, q75


def _log_ticks(counts, ax, axis="y", n=3):
    """Place exactly n log10 tick labels spanning the count range."""
    mx = max(float(np.max(counts)), 1.0)
    hi = int(np.ceil(np.log10(max(mx, 10.0))))
    exps = np.unique(np.round(np.linspace(0, hi, n)).astype(int))
    ticks = (10.0 ** exps).astype(float)
    labels = ["1" if e == 0 else f"$10^{{{e}}}$" for e in exps]
    if axis == "y":
        ax.set_yticks(ticks)
        ax.set_yticklabels(labels, fontsize=4)
        ax.yaxis.set_minor_locator(plt.NullLocator())
    else:
        ax.set_xticks(ticks)
        ax.set_xticklabels(labels, fontsize=4, rotation=0)
        ax.xaxis.set_minor_locator(plt.NullLocator())


shade_kwargs = dict(facecolor="0.88", edgecolor="none", alpha=0.9, zorder=0, lw=0)

fig = plt.figure(figsize=(fig_width, fig_height), dpi=fig_dpi)
gs = fig.add_gridspec(
    3, 1,
    height_ratios=[1.15, 1.0, 1.1],
    hspace=0.55,
)

# ---- a: three joint plots ----
gs_a = gs[0].subgridspec(1, 3, wspace=0.28)
amb_bins = np.linspace(0, 100, 41)
marg_ratio = 0.22
ax_a_mains = []
ax_a_tops = []
ax_a_rights = []
for i, gt in enumerate(ccc_gene_types):
    sub = amb_scatter[amb_scatter["gene_type_grouped"] == gt]
    color = gene_type_palette[gt]
    inner = gs_a[i].subgridspec(
        2, 2, width_ratios=[1, marg_ratio], height_ratios=[marg_ratio, 1],
        wspace=0.05, hspace=0.05,
    )
    ax_sc = fig.add_subplot(inner[1, 0])
    ax_top = fig.add_subplot(inner[0, 0], sharex=ax_sc)
    ax_right = fig.add_subplot(inner[1, 1], sharey=ax_sc)
    ax_a_mains.append(ax_sc)
    ax_a_tops.append(ax_top)
    ax_a_rights.append(ax_right)
    # newly highly ambiguous: v39 ≤ 75%, v47 > 75%
    ax_sc.add_patch(Rectangle(
        (-2, high_amb_pct),
        high_amb_pct - (-2),
        102 - high_amb_pct,
        **shade_kwargs,
    ))
    ax_sc.scatter(
        sub["x"], sub["y"], s=5, facecolors="none", edgecolors=color,
        linewidths=0.3, alpha=0.4, rasterized=True, zorder=2,
    )
    ax_sc.plot([0, 100], [0, 100], ls=":", color="0.5", lw=0.7, zorder=1)
    ax_sc.set_xlim(-2, 102)
    ax_sc.set_ylim(-2, 102)
    # box aspect (not data aspect) so marginals stay aligned with scatter edges
    ax_sc.set_box_aspect(1)
    ax_sc.set_xlabel(r"% ambiguous ($v_{39}$)")
    if i == 0:
        ax_sc.set_ylabel(r"% ambiguous ($v_{47}$)")
    else:
        ax_sc.tick_params(axis="y", labelleft=False)
    sns.despine(ax=ax_sc)

    counts_x, edges_x = np.histogram(sub["x"], bins=amb_bins)
    ax_top.bar(
        edges_x[:-1], np.clip(counts_x, 0.5, None), width=np.diff(edges_x),
        align="edge", color=color, edgecolor="k", linewidth=0.1,
    )
    ax_top.set_yscale("log")
    ax_top.set_ylim(bottom=0.5, top=max(counts_x.max() * 1.5, 10))
    ax_top.tick_params(axis="x", labelbottom=False, length=0)
    _log_ticks(counts_x, ax_top, axis="y")
    ax_top.set_ylabel("genes", fontsize=5)
    ax_top.set_title(gene_type_pretty[gt], fontsize=7, pad=2)
    sns.despine(ax=ax_top, bottom=True)

    counts_y, edges_y = np.histogram(sub["y"], bins=amb_bins)
    ax_right.barh(
        edges_y[:-1], np.clip(counts_y, 0.5, None), height=np.diff(edges_y),
        align="edge", color=color, edgecolor="k", linewidth=0.1,
    )
    ax_right.set_xscale("log")
    ax_right.set_xlim(left=0.5, right=max(counts_y.max() * 1.5, 10))
    ax_right.tick_params(axis="y", labelleft=False, length=0)
    _log_ticks(counts_y, ax_right, axis="x")
    ax_right.set_xlabel("genes", fontsize=5)
    sns.despine(ax=ax_right, left=True)

# ---- middle: b (Δ) + c/d (hexbins) ----
gs_mid = gs[1].subgridspec(1, 2, width_ratios=[1, 2], wspace=0.38)

# ---- b: Δ % ambiguous ----
ax_b = fig.add_subplot(gs_mid[0, 0])
for gt in ccc_gene_types:
    sub = amb_scatter[amb_scatter["gene_type_grouped"] == gt]
    counts = (
        sub["delta_bin"].value_counts()
        .reindex(delta_bin_labels, fill_value=0)
        .to_numpy(dtype=float)
    )
    ax_b.plot(
        delta_bin_centers, np.clip(counts, 0.5, None),
        color=gene_type_palette[gt], lw=0.9,
        label=gene_type_pretty[gt], drawstyle="steps-mid",
    )
ax_b.axvline(0, color="0.6", lw=0.5, ls=":")
ax_b.set_yscale("log")
ax_b.set_ylim(bottom=0.5)
ax_b.set_xlim(-100, 100)
ax_b.set_xticks([-100, 0, 100])
_b_lab_fs = plt.rcParams["axes.labelsize"]
ax_b.set_xlabel(r"$\Delta$ % ambiguous ($v_{47}-v_{39}$)", fontsize=_b_lab_fs)
ax_b.set_ylabel("Genes", fontsize=_b_lab_fs)
ax_b.legend(frameon=False, fontsize=4, loc="upper right")
sns.despine(ax=ax_b)

# ---- c/d: hexbins ----
gs_hex = gs_mid[0, 1].subgridspec(1, 2, wspace=0.45)
_lim_vals = [stranded_v47_mat.max(), stranded_v39_mat.max(), v47_mat.max(), v39_mat.max()]
lim_hi = float(np.ceil(max(_lim_vals) * 10) / 10)
tick_vals = np.array([0.0, 2.0, 4.0])
tick_labs = ["0", "2", "4"]
ax_c = fig.add_subplot(gs_hex[0, 0])  # v39 hexbin
ax_d = fig.add_subplot(gs_hex[0, 1], sharex=ax_c, sharey=ax_c)  # v47 hexbin
hb_by_ax = []
for ax_h, version in zip([ax_c, ax_d], ("v39", "v47")):
    idx = scatter_gene_idx_pooled
    s_mat = _matched_stranded(version)
    hb = ax_h.hexbin(
        s_mat[:, idx].ravel(), unstranded_mat[version][:, idx].ravel(),
        gridsize=40, bins="log", mincnt=1, cmap="Greys",
        extent=(0, lim_hi, 0, lim_hi), linewidths=0, rasterized=True,
    )
    hb_by_ax.append((ax_h, hb))
    ax_h.plot([0, lim_hi], [0, lim_hi], ls=":", color="0.4", lw=0.7, zorder=2)
    med = ccc_pooled.get(version, np.nan)
    if np.isfinite(med):
        ax_h.text(0.04, 0.96, f"CCC={med:.3f}", transform=ax_h.transAxes, va="top", ha="left", fontsize=5)
    ax_h.set_xlim(0, lim_hi)
    ax_h.set_ylim(0, lim_hi)
    ax_h.set_aspect("equal", adjustable="box")
    sns.despine(ax=ax_h)
    ax_h.set_xticks(tick_vals)
    ax_h.set_yticks(tick_vals)
    ax_h.set_xticklabels(tick_labs)
    if version == "v39":
        ax_h.set_xlabel(r"$v_{39}$ stranded $\log_{10}$(TPM+1)")
        ax_h.set_ylabel(r"$v_{39}$ unstranded $\log_{10}$(TPM+1)")
        ax_h.set_yticklabels(tick_labs)
    else:
        ax_h.set_xlabel(r"$v_{47}$ stranded $\log_{10}$(TPM+1)")
        ax_h.set_ylabel(r"$v_{47}$ unstranded $\log_{10}$(TPM+1)")
        ax_h.tick_params(axis="y", labelleft=False)

if hb_by_ax:
    vmax = max(hb.get_array().max() for _, hb in hb_by_ax)
    for _, hb in hb_by_ax:
        hb.set_clim(vmin=1, vmax=vmax)
    fig.canvas.draw()
    for ax_h, hb in hb_by_ax:
        pos = ax_h.get_position()
        cbar_h = 0.5 * pos.height
        cax = fig.add_axes([pos.x1 + 0.006, pos.y0 + 0.25 * pos.height, 0.007, cbar_h])
        cb = fig.colorbar(hb, cax=cax)
        cb.set_label("genes", fontsize=4, labelpad=1)
        cb.ax.yaxis.set_major_locator(plt.LogLocator(base=10, numticks=3))
        cb.ax.yaxis.set_minor_locator(plt.NullLocator())
        cb.ax.tick_params(labelsize=3, pad=1)
        cb.ax.yaxis.set_tick_params(which="both", length=2)

# ---- bottom: e (CCC vs ambig by TPM) + f (newly highly ambiguous) ----
gs_bot = gs[2].subgridspec(1, 3, width_ratios=[1.0, 1.0, 1.05], wspace=0.30)
axes_e = []
for j, q in enumerate(expr_bin_labels):
    ax = fig.add_subplot(gs_bot[0, j], sharey=axes_e[0] if axes_e else None)
    axes_e.append(ax)
    sub = gene_ccc[gene_ccc["expr_bin"] == q]
    ax.scatter(
        sub["pct_ambig_v47"], sub["ccc_v47"],
        s=4, facecolors="none", edgecolors="0.65",
        linewidths=0.25, alpha=0.3, rasterized=True,
    )
    xs, med, q25, q75 = _rolling_ccc_trend(sub["pct_ambig_v47"], sub["ccc_v47"])
    if len(xs):
        ax.fill_between(xs, q25, q75, color=ccc_color_v47, alpha=0.18, lw=0, zorder=2)
        ax.plot(xs, med, color=ccc_color_v47, lw=1.0, zorder=3, label=r"$v_{47}$")
    ax.scatter(
        sub["pct_ambig_v39"], sub["ccc_v39"],
        s=4, facecolors="none", edgecolors=ccc_color_v39,
        linewidths=0.25, alpha=0.2, rasterized=True, label=r"$v_{39}$",
    )
    xs39, med39, q25_39, q75_39 = _rolling_ccc_trend(sub["pct_ambig_v39"], sub["ccc_v39"])
    if len(xs39):
        ax.fill_between(xs39, q25_39, q75_39, color=ccc_color_v39, alpha=0.12, lw=0, zorder=2)
        ax.plot(xs39, med39, color=ccc_color_v39, ls="--", lw=1.0, zorder=3)
    ax.axvline(high_amb_pct, color="0.35", lw=0.7, ls="--", zorder=1)
    ax.set_title(f"{q} median TPM", fontsize=5.5)
    ax.set_xlim(-2, 102)
    ax.set_xticks([0, 50, 75, 100])
    ax.set_xticklabels(["0", "50", "75", "100"])
    ax.set_ylim(0, 1.02)
    ax.set_xlabel(r"% ambiguous")
    if j == 0:
        ax.set_ylabel(r"CCC unstr. vs stranded")
        ax.legend(frameon=False, fontsize=4, loc="lower left")
    else:
        ax.tick_params(axis="y", labelleft=False)
    sns.despine(ax=ax)

# f: newly highly ambiguous genes — CCC under v39 vs v47
ax_f = fig.add_subplot(gs_bot[0, 2])
nh = gene_ccc.loc[gene_ccc["newly_high_amb"]].copy()
for gt in ccc_gene_types:
    sub = nh[nh["gene_type_grouped"] == gt]
    if sub.empty:
        continue
    ax_f.scatter(
        sub["ccc_v39"], sub["ccc_v47"],
        s=8, facecolors="none", edgecolors=gene_type_palette[gt],
        linewidths=0.45, alpha=0.55, rasterized=True,
        label=gene_type_pretty[gt],
    )
ax_f.plot([0, 1], [0, 1], ls=":", color="0.5", lw=0.7, zorder=0)
ax_f.set_xlim(-0.02, 1.02)
ax_f.set_ylim(-0.02, 1.02)
ax_f.set_aspect("equal", adjustable="box")
ax_f.set_xlabel(r"CCC ($v_{39}$)")
ax_f.set_ylabel(r"CCC ($v_{47}$)")
_med39 = float(np.nanmedian(nh["ccc_v39"])) if len(nh) else np.nan
_med47 = float(np.nanmedian(nh["ccc_v47"])) if len(nh) else np.nan
ax_f.set_title(
    f"newly >{high_amb_pct:g}% ambiguous\n(no coordinate change)\n"
    f"(n={len(nh):,}; med {_med39:.2f} → {_med47:.2f})",
    fontsize=5.5,
)
ax_f.legend(frameon=False, fontsize=4, loc="upper left")
sns.despine(ax=ax_f)

# panel letters: a, b Δ, c/d hexbins, e CCC-by-expr, f newly high amb
_inv = fig.transFigure.inverted()
_lab_specs = [
    (ax_a_mains[0], "a", -0.18, 1.35),
    (ax_b, "b", -0.22, 1.12),
    (ax_c, "c", -0.18, 1.12),
    (ax_d, "d", -0.18, 1.12),
    (axes_e[0], "e", -0.22, 1.12),
    (ax_f, "f", -0.18, 1.12),
]
for _ax, _lab, dx, dy in _lab_specs:
    _xy = _inv.transform(_ax.transAxes.transform((dx, dy)))
    fig.text(_xy[0], _xy[1], _lab, fontweight="bold", va="bottom", ha="left", fontsize=9)

# Align panel-a marginals to scatter boxes after full layout
fig.canvas.draw()
for ax_sc, ax_top, ax_right in zip(ax_a_mains, ax_a_tops, ax_a_rights):
    pos = ax_sc.get_position()
    ptop = ax_top.get_position()
    pright = ax_right.get_position()
    ax_top.set_position([pos.x0, ptop.y0, pos.width, ptop.height])
    ax_right.set_position([pright.x0, pos.y0, pright.width, pos.height])

plt.show()


## Save figure


In [ ]:
fig_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_pdf, bbox_inches="tight")
fig.savefig(fig_png, bbox_inches="tight")
print(f"Wrote {fig_pdf}")
print(f"Wrote {fig_png}")


## Print stats


In [ ]:
# Numbers cited for Supplementary Fig. 2 (caption c,d,f + Results)
print(f"pooled v39 stranded-unstranded CCC = {ccc_pooled['v39']:.3f}")
print(f"pooled v47 stranded-unstranded CCC = {ccc_pooled['v47']:.3f}")

nh = gene_ccc.loc[gene_ccc["newly_high_amb"]]
print(
    f"newly highly ambiguous genes (>{high_amb_pct:g}% in v47, ≤{high_amb_pct:g}% in v39, "
    f"unchanged coordinates): n = {len(nh):,}"
)
